[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abpandey-dev/learn-python-from-scratch-to-advanced/blob/main/notebooks/10-engineering-toolbox/04_packaging.ipynb)

*Engineering Toolbox · Notebook 4 of 8*

# Packages, Versions & Environments

What pip, uv and poetry do for you: comparing versions, matching specifiers, picking and resolving versions, reading pyproject.toml, parsing requirements and markers, wheel tags, lockfile diffs, vulnerability checks, dependency trees, conflicts, configuration precedence, .env files, PATH lookup, pinning policy, safe upgrades, install size and semantic-version API checks.

### 👩‍🏫 From your teacher

**The picture to keep in your head:** A lockfile is a shopping receipt: it lists exactly which version of everything you bought, so anyone can buy the same again.

**What interviewers really test here:** Virtual environments, pyproject.toml vs lockfiles, version specifiers, dependency conflicts, vulnerability checks and keeping config and secrets out of code.

### How to use this notebook

1. **Run the setup cell** just below.
2. **Read each lesson slowly.** Work the numbers on paper before reading on.
3. **Run and then change the examples.** Breaking code on purpose is how you learn.
4. **Try each practice question before opening an answer.** If you're stuck, open one hint at a time.
5. **Say your interview answers out loud.** Knowing something and explaining it are different skills.

### Your roadmap

- [ ] 1. Environments and dependencies *(Basics)*
- [ ] 2. Versions and specifiers *(Core)*
- [ ] 3. How installers resolve and install *(Core)*
- [ ] 4. Keeping dependencies healthy and config clean *(Advanced)*
- [ ] 20 practice questions
- [ ] Interview questions, out loud

In [1]:
# ▶ Run this cell first. It prepares the notebook; you don't need to read it.
import contextlib, copy, io, math, sys
import numpy as np, matplotlib.pyplot as plt
%matplotlib inline
import math
import json
PLAIN = True

class ListNode:
    def __init__(self, val=0, next=None):
        self.val, self.next = val, next

class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def mk_list(vals, pos=-1):
    dummy = cur = ListNode()
    nodes = []
    for v in vals:
        cur.next = ListNode(v)
        cur = cur.next
        nodes.append(cur)
    if pos >= 0 and nodes:
        nodes[-1].next = nodes[pos]
    return dummy.next

def list_arr(head):
    out, seen = [], set()
    while head and id(head) not in seen:
        seen.add(id(head))
        out.append(head.val)
        head = head.next
    return out

def mk_tree(vals):
    if not vals or vals[0] is None:
        return None
    root = TreeNode(vals[0])
    queue, i = [root], 1
    while queue and i < len(vals):
        node = queue.pop(0)
        for side in ("left", "right"):
            if i < len(vals) and vals[i] is not None:
                child = TreeNode(vals[i])
                setattr(node, side, child)
                queue.append(child)
            i += 1
    return root

def tree_arr(root):
    out, queue = [], [root]
    while queue:
        node = queue.pop(0)
        if node:
            out.append(node.val)
            queue += [node.left, node.right]
        else:
            out.append(None)
    while out and out[-1] is None:
        out.pop()
    return out

def find(root, val):
    if not root or root.val == val:
        return root
    return find(root.left, val) or find(root.right, val)

ListNode.__repr__ = lambda self: f"ListNode{list_arr(self)}"
TreeNode.__repr__ = lambda self: f"TreeNode{tree_arr(self)}"

def plain(x, digits=4):
    """NumPy / pandas results as plain Python values, so they compare with a test's expected answer."""
    np = sys.modules.get("numpy")
    pd = sys.modules.get("pandas")
    if pd is not None:
        if isinstance(x, pd.DataFrame):
            return {str(c): plain(x[c].tolist(), digits) for c in x.columns}
        if isinstance(x, pd.Series):
            return {plain(k, digits): plain(v, digits) for k, v in x.items()}
        if isinstance(x, pd.Index):
            return plain(x.tolist(), digits)
        if x is pd.NaT:
            return None
        if isinstance(x, pd.Timestamp):
            return str(x.date()) if x == x.normalize() else str(x)
    if np is not None:
        if isinstance(x, np.ndarray):
            return plain(x.tolist(), digits)
        if isinstance(x, np.generic):
            return plain(x.item(), digits)
    if isinstance(x, float):
        return None if x != x else round(x, digits) + 0.0
    if isinstance(x, list):
        return [plain(v, digits) for v in x]
    if isinstance(x, tuple):
        return tuple(plain(v, digits) for v in x)
    if isinstance(x, dict):
        return {plain(k, digits): plain(v, digits) for k, v in x.items()}
    if isinstance(x, (set, frozenset)):
        return {plain(v, digits) for v in x}
    return x


def check(fn, tests, plain_compare=None, listify=False, design=False, prep=None, post=None,
          norm=None, verify=None, runner=None, inplace=None):
    """Run your code against the tests and show which pass."""
    use_plain = PLAIN if plain_compare is None else plain_compare
    passed, problems = 0, []
    for args, want in tests:
        try:
            a = copy.deepcopy(args)
            if design:
                obj = fn(*a[0])
                got = [getattr(obj, m)(*p) for m, p in a[1]]
            elif runner is not None:
                got = runner(fn, a)
            else:
                if prep is not None:
                    a = prep(a)
                with contextlib.redirect_stdout(io.StringIO()):
                    got = fn(*a)
                if inplace is not None:
                    got = a[inplace]
                if listify:
                    got = list(got)
                if post is not None:
                    got = post(got)
            if verify is not None:
                ok = bool(verify(copy.deepcopy(args), got))
            elif norm is not None:
                ok = norm(copy.deepcopy(got)) == norm(copy.deepcopy(want))
            else:
                ok = (plain(got), plain(want)) if use_plain else (got, want)
                ok = ok[0] == ok[1]
            ok = ok if isinstance(ok, bool) else False
        except Exception as e:  # your code raised an error: report it, don't stop the notebook
            ok, got = False, f"{type(e).__name__}: {e}"
        passed += ok
        if not ok:
            problems.append((args, want, got))
    plt.close("all")
    if passed == len(tests):
        print(f"✅ All {len(tests)} tests passed. Well done! Now compare with the answers below.")
    else:
        print(f"❌ {passed}/{len(tests)} tests passed. First one to fix:")
        args, want, got = problems[0]
        print("   input:   ", repr(args)[:300])
        print("   expected:", repr(want)[:300])
        print("   you got: ", repr(got)[:300])
    return None

print("Ready! Scroll down and start learning.")

Ready! Scroll down and start learning.


---
# Part 1: The theory in one place

Read this once for the big picture. It's fine if it doesn't all stick yet: the lessons will make it concrete, and you can come back to revise.

### Tools at a glance

| Tool | Does |
| --- | --- |
| pip + venv | install packages into an environment (built in) |
| uv | very fast all-in-one: environments, resolving, lockfiles, Python versions |
| poetry / pdm | project management with lockfiles |
| conda / mamba | also manages non-Python libraries (CUDA, GDAL) |
| pip-audit / osv-scanner | known-vulnerability checks |

---
# Part 2: Lessons, step by step

## Lesson 1: Environments and dependencies

*Level: Basics*

Every project gets its own **virtual environment** — a folder with its own Python and packages — so projects can't break each other. Dependencies are declared in `pyproject.toml`; a **lockfile** records the exact versions of everything installed, so every machine and every CI run gets the same set.



```python
uv init invoice-bot && cd invoice-bot   # pyproject.toml + .venv
uv add fastapi "pydantic>=2,<3"         # updates pyproject.toml and uv.lock
uv add --dev pytest ruff mypy
uv run pytest                           # run inside the environment
uv sync --frozen                        # CI: install exactly the lockfile

# the classic way
python -m venv .venv && source .venv/bin/activate   # Windows: .venv\Scripts\activate
pip install -r requirements.txt
```



Activating an environment just puts its `bin` folder first on `PATH` — that's why `python` suddenly means a different program.

**✏️ Your turn:** you now have what you need for [Q3](#q3), [Q5](#q5), [Q7](#q7). Try one now, or carry on and do them in Part 3.

## Lesson 2: Versions and specifiers

*Level: Core*

Versions compare number by number (`1.10 > 1.9`), and pre-releases (`2.0.0rc1`) come before the final release and aren't installed unless you ask. Specifiers say which versions you accept:



| Specifier | Means | Use for |
| --- | --- | --- |
| `==1.4.2` | exactly that | lockfiles, reproducible apps |
| `>=2,<3` | any 2.x | libraries: allow fixes, block breaking changes |
| `~=1.4.2` | ≥1.4.2 and 1.4.\* | patch updates only |
| `!=1.5.0` | skip a broken release | known bugs |



Rule of thumb: **applications pin** (via a lockfile), **libraries use ranges** so they don't fight with other libraries.

**✏️ Your turn:** you now have what you need for [Q1](#q1), [Q8](#q8), [Q2](#q2), [Q9](#q9), [Q10](#q10), [Q6](#q6). Try one now, or carry on and do them in Part 3.

## Lesson 3: How installers resolve and install

*Level: Core*

The **resolver** picks one version of every package so that all constraints hold at once — a search that backtracks when a choice leads to a conflict. If no combination works, you get a resolution error naming the packages that disagree. Installers then download **wheels** — pre-built archives whose filename tags say which Python, ABI and platform they work on — and fall back to building from source if none match.



```python
uv tree                      # what depends on what
uv tree --invert --package idna   # who pulls in idna
pip install "pydantic<2" "langchain>=0.3"   # → ResolutionImpossible: shows the conflict
```

**✏️ Your turn:** you now have what you need for [Q20](#q20), [Q15](#q15), [Q11](#q11), [Q13](#q13), [Q14](#q14). Try one now, or carry on and do them in Part 3.

## Lesson 4: Keeping dependencies healthy and config clean

*Level: Advanced*

- Review **lockfile diffs** in PRs; let **Dependabot** or **Renovate** open small upgrade PRs (patches automatically, majors reviewed).
- Run **pip-audit** or **osv-scanner** in CI to catch packages with known vulnerabilities.
- Publishing a library? Follow **semantic versioning**: any breaking API change is a major bump.
- **Configuration** belongs outside the code (twelve-factor): defaults in code, overridden by a file, environment variables, then command-line flags. Secrets live in environment variables or a secret manager — never in Git (add `.env` to `.gitignore`).



```python
from pydantic_settings import BaseSettings
class Settings(BaseSettings):
    db_url: str = "sqlite:///dev.db"
    workers: int = 1
    model_config = {"env_prefix": "APP_", "env_file": ".env"}
settings = Settings()        # reads APP_DB_URL, APP_WORKERS from env or .env
```

**✏️ Your turn:** you now have what you need for [Q4](#q4), [Q12](#q12), [Q18](#q18), [Q19](#q19), [Q16](#q16), [Q17](#q17). Try one now, or carry on and do them in Part 3.

---
# Part 3: Practice like an interview

There are 20 questions here, easiest first.

**Work the way you would in a real interview:** restate the problem, try an example by hand, say the simple approach and its cost, improve it, then write the code. Test it with the cell. Only open an answer once you've made a real attempt, because struggling is where the learning happens.

**▶ Run this cell before the questions.** It sets up the data and tools they share.

In [2]:
import re

PRE = {"a": 0, "b": 1, "rc": 2}

import itertools


<a id="q1"></a>
### Q1. Sort Versions Correctly

🟢 Warm-up · *1.10 comes after 1.9*

Sort version strings from oldest to newest. Compare release numbers as integers (missing parts count as 0), and a pre-release (`a` alpha &lt; `b` beta &lt; `rc`) comes before the final release. String sorting gets this wrong.

**Examples**

- `versions = ['1.10.0', '1.9.3', '2.0.0', '2.0.0rc1', '1.9', '2.0.0a2', '2.0.0b1', '1.9.10', '0.12']` → `['0.12', '1.9', '1.9.3', '1.9.10', '1.10.0', '2.0.0a2', '2.0.0b1', '2.0.0rc1', '2.0.0']`

In [ ]:
def sort_versions(versions):
    """
    Sort version strings from oldest to newest. Compare release numbers as
    integers (missing parts count as 0), and a pre-release (a alpha b beta rc)
    comes before the final release. String sorting gets this wrong.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(sort_versions, [
    ((['1.10.0', '1.9.3', '2.0.0', '2.0.0rc1', '1.9', '2.0.0a2', '2.0.0b1', '1.9.10', '0.12'],), ['0.12', '1.9', '1.9.3', '1.9.10', '1.10.0', '2.0.0a2', '2.0.0b1', '2.0.0rc1', '2.0.0']),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Insertion sort with a hand-written compare.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`sorted` with the `vkey` tuple.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Insertion sort with a hand-written compare.

```python
def vkey(v):
    m = re.fullmatch(r"(\d+(?:\.\d+)*)(?:(a|b|rc)(\d+))?", v)
    release = [int(x) for x in m.group(1).split(".")]
    release += [0] * (4 - len(release))
    pre = (PRE[m.group(2)], int(m.group(3))) if m.group(2) else (3, 0)
    return (tuple(release), pre)


def sort_versions(versions):
    out = []
    for v in versions:
        i = 0
        while i < len(out) and vkey(out[i]) <= vkey(v):
            i += 1
        out.insert(i, v)
    return out
```

**Cost:** O(n²) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** `sorted` with the `vkey` tuple.

```python
def vkey(v):
    m = re.fullmatch(r"(\d+(?:\.\d+)*)(?:(a|b|rc)(\d+))?", v)
    release = [int(x) for x in m.group(1).split(".")]
    release += [0] * (4 - len(release))
    pre = (PRE[m.group(2)], int(m.group(3))) if m.group(2) else (3, 0)
    return (tuple(release), pre)


def sort_versions(versions):
    return sorted(versions, key=vkey)
```

**Cost:** O(n log n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Insertion sort with a hand-written compare. That costs O(n²) time and O(n) space."

"I can do better: `sorted` with the `vkey` tuple. That's O(n log n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/engineering-toolbox/#/q/packaging/sort-versions-correctly)

<a id="q2"></a>
### Q2. Pick the Newest Allowed Version

🟢 Warm-up · *What pip install chooses*

Return the newest version that satisfies the specifier for each `(package, spec)`, skipping pre-releases unless `allow_pre` is True. Return None if nothing matches.

**Examples**

- `available = {'requests': ['2.30.0', '2.31.0', '2.32.3', '3.0.0b1'], 'numpy': ['1.26.4', '2.0.0', '2.1.3', '2.2.0rc1'], 'django': ['4.2.16', '5.0.9', '5.1.3']}, wanted = [('requests', '>=2.31,<4'), ('numpy', '<2'), ('django', '>=6')], allow_pre = False` → `{'requests': '2.32.3', 'numpy': '1.26.4', 'django': None}`
- `available = {'requests': ['2.30.0', '2.31.0', '2.32.3', '3.0.0b1'], 'numpy': ['1.26.4', '2.0.0', '2.1.3', '2.2.0rc1'], 'django': ['4.2.16', '5.0.9', '5.1.3']}, wanted = [('requests', '>=2.31,<4'), ('numpy', '>=2')], allow_pre = True` → `{'requests': '3.0.0b1', 'numpy': '2.2.0rc1'}`

In [ ]:
def newest_allowed(available, wanted, allow_pre):
    """
    Return the newest version that satisfies the specifier for each (package,
    spec), skipping pre-releases unless allow_pre is True. Return None if
    nothing matches.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(newest_allowed, [
    (({'requests': ['2.30.0', '2.31.0', '2.32.3', '3.0.0b1'], 'numpy': ['1.26.4', '2.0.0', '2.1.3', '2.2.0rc1'], 'django': ['4.2.16', '5.0.9', '5.1.3']}, [('requests', '>=2.31,<4'), ('numpy', '<2'), ('django', '>=6')], False), {'requests': '2.32.3', 'numpy': '1.26.4', 'django': None}),
    (({'requests': ['2.30.0', '2.31.0', '2.32.3', '3.0.0b1'], 'numpy': ['1.26.4', '2.0.0', '2.1.3', '2.2.0rc1'], 'django': ['4.2.16', '5.0.9', '5.1.3']}, [('requests', '>=2.31,<4'), ('numpy', '>=2')], True), {'requests': '3.0.0b1', 'numpy': '2.2.0rc1'}),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Filter, then find the max by comparing keys in a loop.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`max` with a key and a default.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Filter, then find the max by comparing keys in a loop.

```python
def vkey(v):
    m = re.fullmatch(r"(\d+(?:\.\d+)*)(?:(a|b|rc)(\d+))?", v)
    release = [int(x) for x in m.group(1).split(".")]
    release += [0] * (4 - len(release))
    pre = (PRE[m.group(2)], int(m.group(3))) if m.group(2) else (3, 0)
    return (tuple(release), pre)

def is_pre(v):
    return vkey(v)[1][0] < 3

def satisfies(v, spec):
    for part in [p.strip() for p in spec.split(",") if p.strip()]:
        op, target = re.fullmatch(r"(~=|==|!=|>=|<=|>|<)\s*(\S+)", part).groups()
        if target.endswith(".*"):
            prefix = target[:-2].split(".")
            same = [str(x) for x in vkey(v)[0][:len(prefix)]] == prefix
            if (op == "==") != same:
                return False
            continue
        a, b = vkey(v), vkey(target)
        if op == "~=":
            head = target.split(".")[:-1]
            if not (a >= b and [str(x) for x in a[0][:len(head)]] == head):
                return False
        elif not {"==": a == b, "!=": a != b, ">=": a >= b, "<=": a <= b, ">": a > b, "<": a < b}[op]:
            return False
    return True


def newest_allowed(available, wanted, allow_pre):
    out = {}
    for pkg, spec in wanted:
        best = None
        for v in available[pkg]:
            if is_pre(v) and not allow_pre:
                continue
            if satisfies(v, spec) and (best is None or vkey(v) > vkey(best)):
                best = v
        out[pkg] = best
    return out
```

**Cost:** O(r · v) time, O(v) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** `max` with a key and a default.

```python
def vkey(v):
    m = re.fullmatch(r"(\d+(?:\.\d+)*)(?:(a|b|rc)(\d+))?", v)
    release = [int(x) for x in m.group(1).split(".")]
    release += [0] * (4 - len(release))
    pre = (PRE[m.group(2)], int(m.group(3))) if m.group(2) else (3, 0)
    return (tuple(release), pre)

def is_pre(v):
    return vkey(v)[1][0] < 3

def satisfies(v, spec):
    for part in [p.strip() for p in spec.split(",") if p.strip()]:
        op, target = re.fullmatch(r"(~=|==|!=|>=|<=|>|<)\s*(\S+)", part).groups()
        if target.endswith(".*"):
            prefix = target[:-2].split(".")
            same = [str(x) for x in vkey(v)[0][:len(prefix)]] == prefix
            if (op == "==") != same:
                return False
            continue
        a, b = vkey(v), vkey(target)
        if op == "~=":
            head = target.split(".")[:-1]
            if not (a >= b and [str(x) for x in a[0][:len(head)]] == head):
                return False
        elif not {"==": a == b, "!=": a != b, ">=": a >= b, "<=": a <= b, ">": a > b, "<": a < b}[op]:
            return False
    return True


def newest_allowed(available, wanted, allow_pre):
    return {p: max((v for v in available[p] if satisfies(v, s) and (allow_pre or not is_pre(v))), key=vkey, default=None)
            for p, s in wanted}
```

**Cost:** O(r · v) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Filter, then find the max by comparing keys in a loop. That costs O(r · v) time and O(v) space."

"I can do better: `max` with a key and a default. That's O(r · v) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/engineering-toolbox/#/q/packaging/pick-the-newest-allowed-version)

<a id="q3"></a>
### Q3. Read pyproject.toml

🟢 Warm-up · *The modern project file*

Parse the TOML text with `tomllib` and return `{'name', 'version', 'requires_python', 'dependencies' (count), 'dev' (count of the 'dev' optional group, 0 if absent), 'scripts' (sorted names)}`.

**Examples**

- `text = '[project]\nname = "invoice-bot"\nversion = "0.3.1"\nrequires-python = ">=3.11"\ndependencies = [\n  "fastapi>=0.115",\n  "pydantic>=2,<3",\n  "httpx",\n]\n\n[project.optional-dependencies]\ndev = ["pytest", "ruff", "mypy"]\n\n[project.scripts]\ninvoice-bot = "invoice_bot.cli:main"\ninvoice-admin = "invoice_bot.admin:main"\n'` → `{'name': 'invoice-bot', 'version': '0.3.1', 'requires_python': '>=3.11', 'dependencies': 3, 'dev': 3, 'scripts': ['invoice-admin', 'invoice-bot']}`

In [ ]:
import tomllib

def read_pyproject(text):
    """
    Parse the TOML text with tomllib and return {'name', 'version',
    'requires_python', 'dependencies' (count), 'dev' (count of the 'dev'
    optional group, 0 if absent), 'scripts' (sorted names)}.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(read_pyproject, [
    (('[project]\nname = "invoice-bot"\nversion = "0.3.1"\nrequires-python = ">=3.11"\ndependencies = [\n  "fastapi>=0.115",\n  "pydantic>=2,<3",\n  "httpx",\n]\n\n[project.optional-dependencies]\ndev = ["pytest", "ruff", "mypy"]\n\n[project.scripts]\ninvoice-bot = "invoice_bot.cli:main"\ninvoice-admin = "invoice_bot.admin:main"\n',), {'name': 'invoice-bot', 'version': '0.3.1', 'requires_python': '>=3.11', 'dependencies': 3, 'dev': 3, 'scripts': ['invoice-admin', 'invoice-bot']}),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Parse, then read keys step by step.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Parse, then `dict.get` chains.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Parse, then read keys step by step.

```python
def read_pyproject(text):
    data = tomllib.loads(text)
    p = data["project"]
    dev = 0
    if "optional-dependencies" in p and "dev" in p["optional-dependencies"]:
        dev = len(p["optional-dependencies"]["dev"])
    scripts = []
    if "scripts" in p:
        scripts = sorted(p["scripts"])
    return {"name": p["name"], "version": p["version"], "requires_python": p.get("requires-python"),
            "dependencies": len(p.get("dependencies", [])), "dev": dev, "scripts": scripts}
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Parse, then `dict.get` chains.

```python
def read_pyproject(text):
    p = tomllib.loads(text)["project"]
    return {"name": p["name"], "version": p["version"], "requires_python": p.get("requires-python"),
            "dependencies": len(p.get("dependencies", [])), "dev": len(p.get("optional-dependencies", {}).get("dev", [])),
            "scripts": sorted(p.get("scripts", {}))}
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Parse, then read keys step by step. That costs O(n) time and O(n) space."

"I can do better: Parse, then `dict.get` chains. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/engineering-toolbox/#/q/packaging/read-pyproject-toml)

<a id="q4"></a>
### Q4. What Changed in the Lockfile?

🟢 Warm-up · *Review dependency updates*

Compare two lockfiles (package → version) and return `{'added', 'removed', 'upgraded', 'downgraded'}`, each a sorted list (changes written as `'pkg old -> new'`).

**Examples**

- `old = {'fastapi': '0.110.0', 'starlette': '0.36.3', 'pydantic': '2.6.4', 'anyio': '4.3.0', 'sniffio': '1.3.1'}, new = {'fastapi': '0.115.4', 'starlette': '0.41.2', 'pydantic': '2.6.4', 'anyio': '4.2.0', 'typing-extensions': '4.12.2'}` → `{'added': ['typing-extensions'], 'removed': ['sniffio'], 'upgraded': ['fastapi 0.110.0 -> 0.115.4', 'starlette 0.36.3 -> 0.41.2'], 'downgraded': ['anyio 4.3.0 -> 4.2.0']}`

In [ ]:
def lock_diff(old, new):
    """
    Compare two lockfiles (package → version) and return {'added', 'removed',
    'upgraded', 'downgraded'}, each a sorted list (changes written as 'pkg old
    -> new').
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(lock_diff, [
    (({'fastapi': '0.110.0', 'starlette': '0.36.3', 'pydantic': '2.6.4', 'anyio': '4.3.0', 'sniffio': '1.3.1'}, {'fastapi': '0.115.4', 'starlette': '0.41.2', 'pydantic': '2.6.4', 'anyio': '4.2.0', 'typing-extensions': '4.12.2'}), {'added': ['typing-extensions'], 'removed': ['sniffio'], 'upgraded': ['fastapi 0.110.0 -> 0.115.4', 'starlette 0.36.3 -> 0.41.2'], 'downgraded': ['anyio 4.3.0 -> 4.2.0']}),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Loop over the union of names.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Set operations and comprehensions.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Loop over the union of names.

```python
def vkey(v):
    m = re.fullmatch(r"(\d+(?:\.\d+)*)(?:(a|b|rc)(\d+))?", v)
    release = [int(x) for x in m.group(1).split(".")]
    release += [0] * (4 - len(release))
    pre = (PRE[m.group(2)], int(m.group(3))) if m.group(2) else (3, 0)
    return (tuple(release), pre)


def lock_diff(old, new):
    out = {"added": [], "removed": [], "upgraded": [], "downgraded": []}
    for p in sorted(set(old) | set(new)):
        if p not in old:
            out["added"].append(p)
        elif p not in new:
            out["removed"].append(p)
        elif vkey(new[p]) > vkey(old[p]):
            out["upgraded"].append(f"{p} {old[p]} -> {new[p]}")
        elif vkey(new[p]) < vkey(old[p]):
            out["downgraded"].append(f"{p} {old[p]} -> {new[p]}")
    return out
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Set operations and comprehensions.

```python
def vkey(v):
    m = re.fullmatch(r"(\d+(?:\.\d+)*)(?:(a|b|rc)(\d+))?", v)
    release = [int(x) for x in m.group(1).split(".")]
    release += [0] * (4 - len(release))
    pre = (PRE[m.group(2)], int(m.group(3))) if m.group(2) else (3, 0)
    return (tuple(release), pre)


def lock_diff(old, new):
    both = sorted(old.keys() & new.keys())
    fmt = lambda p: f"{p} {old[p]} -> {new[p]}"
    return {"added": sorted(new.keys() - old.keys()), "removed": sorted(old.keys() - new.keys()),
            "upgraded": [fmt(p) for p in both if vkey(new[p]) > vkey(old[p])],
            "downgraded": [fmt(p) for p in both if vkey(new[p]) < vkey(old[p])]}
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Loop over the union of names. That costs O(n) time and O(n) space."

"I can do better: Set operations and comprehensions. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/engineering-toolbox/#/q/packaging/what-changed-in-the-lockfile)

<a id="q5"></a>
### Q5. Which python Runs?

🟢 Warm-up · *PATH and virtual environments*

A shell runs the first matching executable along `PATH` (directories in order). Activating a virtual environment just puts its `bin` folder first. For each command, return `(path that runs, other matches that are shadowed)`, or `(None, [])` if not found.

**Examples**

- `path_dirs = ['/home/asha/proj/.venv/bin', '/usr/local/bin', '/usr/bin'], contents = {'/home/asha/proj/.venv/bin': ['python', 'pip', 'pytest'], '/usr/local/bin': ['python', 'uv'], '/usr/bin': ['python', 'git', 'pip']}, commands = ['python', 'git', 'pytest', 'pip', 'docker']` → `{'python': ('/home/asha/proj/.venv/bin/python', ['/usr/local/bin/python', '/usr/bin/python']), 'git': ('/usr/bin/git', []), 'pytest': ('/home/asha/proj/.venv/bin/pytest', []), 'pip': ('/home/asha/proj/.venv/bin/pip', ['/usr/bin/pip']), 'docker': (None, [])}`

In [ ]:
def which(path_dirs, contents, commands):
    """
    A shell runs the first matching executable along PATH (directories in
    order). Activating a virtual environment just puts its bin folder first. For
    each command, return (path that runs, other matches that are shadowed), or
    (None, []) if not found.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(which, [
    ((['/home/asha/proj/.venv/bin', '/usr/local/bin', '/usr/bin'], {'/home/asha/proj/.venv/bin': ['python', 'pip', 'pytest'], '/usr/local/bin': ['python', 'uv'], '/usr/bin': ['python', 'git', 'pip']}, ['python', 'git', 'pytest', 'pip', 'docker']), {'python': ('/home/asha/proj/.venv/bin/python', ['/usr/local/bin/python', '/usr/bin/python']), 'git': ('/usr/bin/git', []), 'pytest': ('/home/asha/proj/.venv/bin/pytest', []), 'pip': ('/home/asha/proj/.venv/bin/pip', ['/usr/bin/pip']), 'docker': (None, [])}),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Scan directories for each command.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A comprehension of matching paths.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Scan directories for each command.

```python
def which(path_dirs, contents, commands):
    out = {}
    for c in commands:
        hits = []
        for d in path_dirs:
            if c in contents.get(d, []):
                hits.append(d + "/" + c)
        out[c] = (hits[0], hits[1:]) if hits else (None, [])
    return out
```

**Cost:** O(c · d) time, O(d) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A comprehension of matching paths.

```python
def which(path_dirs, contents, commands):
    out = {}
    for c in commands:
        hits = [f"{d}/{c}" for d in path_dirs if c in contents.get(d, ())]
        out[c] = (hits[0], hits[1:]) if hits else (None, [])
    return out
```

**Cost:** O(c · d) time, O(d) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Scan directories for each command. That costs O(c · d) time and O(d) space."

"I can do better: A comprehension of matching paths. That's O(c · d) time and O(d) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/engineering-toolbox/#/q/packaging/which-python-runs)

<a id="q6"></a>
### Q6. Pin, Range or Unbounded?

🟢 Warm-up · *Apps pin, libraries range*

Classify each requirement spec: `'pinned'` (a single `==` without `*`), `'compatible'` (has an upper bound: `<`, `<=`, `~=` or `==X.*`), `'lower bound only'` (only `>=`/`>`/`!=`), or `'unbounded'` (empty). Return a dict.

**Examples**

- `specs = {'django': '==5.1.3', 'requests': '>=2.31', 'pydantic': '>=2,<3', 'numpy': '', 'httpx': '~=0.27.0', 'boto3': '==1.35.*'}` → `{'django': 'pinned', 'requests': 'lower bound only', 'pydantic': 'compatible', 'numpy': 'unbounded', 'httpx': 'compatible', 'boto3': 'compatible'}`

In [ ]:
def pin_kind(specs):
    """
    Classify each requirement spec: 'pinned' (a single == without *),
    'compatible' (has an upper bound: <, <=, ~= or ==X.*), 'lower bound only'
    (only >=/>/!=), or 'unbounded' (empty). Return a dict.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(pin_kind, [
    (({'django': '==5.1.3', 'requests': '>=2.31', 'pydantic': '>=2,<3', 'numpy': '', 'httpx': '~=0.27.0', 'boto3': '==1.35.*'},), {'django': 'pinned', 'requests': 'lower bound only', 'pydantic': 'compatible', 'numpy': 'unbounded', 'httpx': 'compatible', 'boto3': 'compatible'}),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

If/elif on the operators present.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A regex for the operators and a lookup.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** If/elif on the operators present.

```python
def pin_kind(specs):
    out = {}
    for name, s in specs.items():
        parts = [p for p in s.split(",") if p]
        if not parts:
            out[name] = "unbounded"
        elif len(parts) == 1 and parts[0].startswith("==") and "*" not in parts[0]:
            out[name] = "pinned"
        elif any(p.startswith(("<", "~=")) or (p.startswith("==") and "*" in p) for p in parts):
            out[name] = "compatible"
        else:
            out[name] = "lower bound only"
    return out
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A regex for the operators and a lookup.

```python
def pin_kind(specs):
    def kind(s):
        if not s:
            return "unbounded"
        if re.fullmatch(r"==[^,*]+", s):
            return "pinned"
        return "compatible" if re.search(r"(^|,)(<|~=|==[^,]*\*)", s) else "lower bound only"
    return {n: kind(s) for n, s in specs.items()}
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: If/elif on the operators present. That costs O(n) time and O(n) space."

"I can do better: A regex for the operators and a lookup. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/engineering-toolbox/#/q/packaging/pin-range-or-unbounded)

<a id="q7"></a>
### Q7. How Big Is the Environment?

🟢 Warm-up · *Shared dependencies count once*

Given installed sizes in MB and the dependency graph, return the total size of installing `top` (each package counted once), rounded to 1, and the size each top-level package would add on its own (itself plus its transitive deps).

**Examples**

- `sizes = {'torch': 780.0, 'numpy': 38.5, 'pandas': 70.2, 'scipy': 110.4, 'sympy': 72.0, 'mpmath': 5.5, 'pytz': 2.4, 'networkx': 9.6}, deps = {'torch': ['numpy', 'sympy', 'networkx'], 'sympy': ['mpmath'], 'pandas': ['numpy', 'pytz'], 'scipy': ['numpy'], 'numpy': [], 'mpmath': [], 'pytz': [], 'networkx': []}, top = ['torch', 'pandas', 'scipy']` → `(1088.6, {'torch': 905.6, 'pandas': 111.1, 'scipy': 148.9})`

In [ ]:
def env_size(sizes, deps, top):
    """
    Given installed sizes in MB and the dependency graph, return the total size
    of installing top (each package counted once), rounded to 1, and the size
    each top-level package would add on its own (itself plus its transitive
    deps).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(env_size, [
    (({'torch': 780.0, 'numpy': 38.5, 'pandas': 70.2, 'scipy': 110.4, 'sympy': 72.0, 'mpmath': 5.5, 'pytz': 2.4, 'networkx': 9.6}, {'torch': ['numpy', 'sympy', 'networkx'], 'sympy': ['mpmath'], 'pandas': ['numpy', 'pytz'], 'scipy': ['numpy'], 'numpy': [], 'mpmath': [], 'pytz': [], 'networkx': []}, ['torch', 'pandas', 'scipy']), (1088.6, {'torch': 905.6, 'pandas': 111.1, 'scipy': 148.9})),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Walk the graph per package with a stack.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A reusable closure helper and set unions.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Walk the graph per package with a stack.

```python
def env_size(sizes, deps, top):
    alone = {}
    everything = set()
    for t in top:
        seen, stack = set(), [t]
        while stack:
            p = stack.pop()
            if p not in seen:
                seen.add(p)
                stack += deps[p]
        alone[t] = round(sum(sizes[p] for p in seen), 1)
        everything |= seen
    return (round(sum(sizes[p] for p in everything), 1), alone)
```

**Cost:** O(t · (n + e)) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A reusable closure helper and set unions.

```python
def env_size(sizes, deps, top):
    def closure(p, seen):
        if p not in seen:
            seen.add(p)
            for d in deps[p]:
                closure(d, seen)
        return seen
    parts = {t: closure(t, set()) for t in top}
    return (round(sum(sizes[p] for p in set().union(*parts.values())), 1), {t: round(sum(sizes[p] for p in s), 1) for t, s in parts.items()})
```

**Cost:** O(t · (n + e)) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Walk the graph per package with a stack. That costs O(t · (n + e)) time and O(n) space."

"I can do better: A reusable closure helper and set unions. That's O(t · (n + e)) time and O(n) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `sizes = {'a': 1.0, 'b': 2.0}, deps = {'a': ['b'], 'b': []}, top = ['a']`

1. closure() returns {'a', 'b'}
2. closure() returns {'a', 'b'}
3. env\_size() returns (3.0, {'a': 3.0})

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/engineering-toolbox/#/q/packaging/how-big-is-the-environment)

<a id="q8"></a>
### Q8. Does the Version Match?

🟡 Interview standard · *Version specifiers*

For each specifier (comma means AND), return which of the versions satisfy it, using `satisfies`-style rules: `==1.4.*` matches any 1.4 release; `~=1.4.2` means ≥1.4.2 and 1.4.\*; `~=1.4` means ≥1.4 and 1.\*.

**Examples**

- `versions = ['1.3.9', '1.4.0', '1.4.2', '1.4.7', '1.5.0', '2.0.0'], specs = ['>=1.4,<2', '~=1.4.2', '~=1.4', '==1.4.*', '!=1.5.0,>1.4.0']` → `{'>=1.4,<2': ['1.4.0', '1.4.2', '1.4.7', '1.5.0'], '~=1.4.2': ['1.4.2', '1.4.7'], '~=1.4': ['1.4.0', '1.4.2', '1.4.7', '1.5.0'], '==1.4.*': ['1.4.0', '1.4.2', '1.4.7'], '!=1.5.0,>1.4.0': ['1.4.2', '1.4.7', '2.0.0']}`

In [ ]:
def matching(versions, specs):
    """
    For each specifier (comma means AND), return which of the versions satisfy
    it, using satisfies-style rules: ==1.4.* matches any 1.4 release; ~=1.4.2
    means ≥1.4.2 and 1.4.*; ~=1.4 means ≥1.4 and 1.*.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(matching, [
    ((['1.3.9', '1.4.0', '1.4.2', '1.4.7', '1.5.0', '2.0.0'], ['>=1.4,<2', '~=1.4.2', '~=1.4', '==1.4.*', '!=1.5.0,>1.4.0']), {'>=1.4,<2': ['1.4.0', '1.4.2', '1.4.7', '1.5.0'], '~=1.4.2': ['1.4.2', '1.4.7'], '~=1.4': ['1.4.0', '1.4.2', '1.4.7', '1.5.0'], '==1.4.*': ['1.4.0', '1.4.2', '1.4.7'], '!=1.5.0,>1.4.0': ['1.4.2', '1.4.7', '2.0.0']}),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Loop specs and versions calling the helper.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A comprehension.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Loop specs and versions calling the helper.

```python
def vkey(v):
    m = re.fullmatch(r"(\d+(?:\.\d+)*)(?:(a|b|rc)(\d+))?", v)
    release = [int(x) for x in m.group(1).split(".")]
    release += [0] * (4 - len(release))
    pre = (PRE[m.group(2)], int(m.group(3))) if m.group(2) else (3, 0)
    return (tuple(release), pre)

def satisfies(v, spec):
    for part in [p.strip() for p in spec.split(",") if p.strip()]:
        op, target = re.fullmatch(r"(~=|==|!=|>=|<=|>|<)\s*(\S+)", part).groups()
        if target.endswith(".*"):
            prefix = target[:-2].split(".")
            same = [str(x) for x in vkey(v)[0][:len(prefix)]] == prefix
            if (op == "==") != same:
                return False
            continue
        a, b = vkey(v), vkey(target)
        if op == "~=":
            head = target.split(".")[:-1]
            if not (a >= b and [str(x) for x in a[0][:len(head)]] == head):
                return False
        elif not {"==": a == b, "!=": a != b, ">=": a >= b, "<=": a <= b, ">": a > b, "<": a < b}[op]:
            return False
    return True


def matching(versions, specs):
    out = {}
    for s in specs:
        ok = []
        for v in versions:
            if satisfies(v, s):
                ok.append(v)
        out[s] = ok
    return out
```

**Cost:** O(s · v) time, O(v) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A comprehension.

```python
def vkey(v):
    m = re.fullmatch(r"(\d+(?:\.\d+)*)(?:(a|b|rc)(\d+))?", v)
    release = [int(x) for x in m.group(1).split(".")]
    release += [0] * (4 - len(release))
    pre = (PRE[m.group(2)], int(m.group(3))) if m.group(2) else (3, 0)
    return (tuple(release), pre)

def satisfies(v, spec):
    for part in [p.strip() for p in spec.split(",") if p.strip()]:
        op, target = re.fullmatch(r"(~=|==|!=|>=|<=|>|<)\s*(\S+)", part).groups()
        if target.endswith(".*"):
            prefix = target[:-2].split(".")
            same = [str(x) for x in vkey(v)[0][:len(prefix)]] == prefix
            if (op == "==") != same:
                return False
            continue
        a, b = vkey(v), vkey(target)
        if op == "~=":
            head = target.split(".")[:-1]
            if not (a >= b and [str(x) for x in a[0][:len(head)]] == head):
                return False
        elif not {"==": a == b, "!=": a != b, ">=": a >= b, "<=": a <= b, ">": a > b, "<": a < b}[op]:
            return False
    return True


def matching(versions, specs):
    return {s: [v for v in versions if satisfies(v, s)] for s in specs}
```

**Cost:** O(s · v) time, O(v) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Loop specs and versions calling the helper. That costs O(s · v) time and O(v) space."

"I can do better: A comprehension. That's O(s · v) time and O(v) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/engineering-toolbox/#/q/packaging/does-the-version-match)

<a id="q9"></a>
### Q9. Parse a Requirement Line

🟡 Interview standard · *name[extras] spec ; marker*

Split each requirement into `{'name' (lower-case, with _ and . turned into -), 'extras' (sorted list), 'spec' (spaces removed), 'marker' (or None)}`, e.g. `Requests[socks] >= 2.31, <3 ; python_version >= '3.9'`.

**Examples**

- `lines = ["Requests[socks] >= 2.31, <3 ; python_version >= '3.9'", 'numpy', 'Pydantic_Settings>=2.4', 'uvicorn[standard,watch]==0.30.6', "typing.extensions; python_version < '3.11'"]` → `[{'name': 'requests', 'extras': ['socks'], 'spec': '>=2.31,<3', 'marker': "python_version >= '3.9'"}, {'name': 'numpy', 'extras': [], 'spec': '', 'marker': None}, {'name': 'pydantic-settings', 'extras': [], 'spec': '>=2.4', 'marker': None}, {'name': 'uvicorn', 'extras': ['standard', 'watch'], 'spec': '==0.30.6', 'marker': None}, {'name': 'typing-extensions', 'extras': [], 'spec': '', 'marker': "python_version < '3.11'"}]`

In [ ]:
REQ = re.compile(r"^\s*([A-Za-z0-9][A-Za-z0-9._-]*)\s*(?:\[([^\]]*)\])?\s*([^;]*?)\s*(?:;\s*(.*))?$")

def parse_requirements(lines):
    """
    Split each requirement into {'name' (lower-case, with _ and . turned into
    -), 'extras' (sorted list), 'spec' (spaces removed), 'marker' (or None)},
    e.g. Requests[socks] >= 2.31, = '3.9'.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(parse_requirements, [
    ((["Requests[socks] >= 2.31, <3 ; python_version >= '3.9'", 'numpy', 'Pydantic_Settings>=2.4', 'uvicorn[standard,watch]==0.30.6', "typing.extensions; python_version < '3.11'"],), [{'name': 'requests', 'extras': ['socks'], 'spec': '>=2.31,<3', 'marker': "python_version >= '3.9'"}, {'name': 'numpy', 'extras': [], 'spec': '', 'marker': None}, {'name': 'pydantic-settings', 'extras': [], 'spec': '>=2.4', 'marker': None}, {'name': 'uvicorn', 'extras': ['standard', 'watch'], 'spec': '==0.30.6', 'marker': None}, {'name': 'typing-extensions', 'extras': [], 'spec': '', 'marker': "python_version < '3.11'"}]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Partition the string by hand.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

One regular expression (`REQ`).

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Partition the string by hand.

```python
def parse_requirements(lines):
    out = []
    for line in lines:
        body, _, marker = line.partition(";")
        body = body.strip()
        extras = []
        if "[" in body:
            name, rest = body.split("[", 1)
            ex, spec = rest.split("]", 1)
            extras = sorted(e.strip() for e in ex.split(",") if e.strip())
        else:
            i = 0
            while i < len(body) and (body[i].isalnum() or body[i] in "._-"):
                i += 1
            name, spec = body[:i], body[i:]
        name = name.strip().lower().replace("_", "-").replace(".", "-")
        out.append({"name": name, "extras": extras, "spec": spec.replace(" ", ""), "marker": marker.strip() or None})
    return out
```

**Cost:** O(n · L) time, O(L) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** One regular expression (`REQ`).

```python
def parse_requirements(lines):
    out = []
    for line in lines:
        name, extras, spec, marker = REQ.match(line).groups()
        out.append({"name": re.sub(r"[_.]", "-", name.lower()), "extras": sorted(e.strip() for e in (extras or "").split(",") if e.strip()),
                    "spec": spec.replace(" ", ""), "marker": marker.strip() if marker else None})
    return out
```

**Cost:** O(n · L) time, O(L) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Partition the string by hand. That costs O(n · L) time and O(L) space."

"I can do better: One regular expression (`REQ`). That's O(n · L) time and O(L) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/engineering-toolbox/#/q/packaging/parse-a-requirement-line)

<a id="q10"></a>
### Q10. Evaluate Environment Markers

🟡 Interview standard · *Install this only on Linux, Python &lt; 3.11*

Markers decide whether a requirement applies here. Support comparisons of a variable with a quoted value, joined by `and`/`or` (and binds tighter). Version variables (names ending in `version`) compare as versions. Return a bool per marker for the environment `env`.

**Examples**

- `marker_list = ["python_version >= '3.10'", "python_version < '3.11' and sys_platform == 'linux'", "sys_platform == 'win32' or platform_machine == 'arm64'", "python_version >= '3.9' and python_version < '3.10' or sys_platform == 'darwin'"], env = {'python_version': '3.12', 'sys_platform': 'darwin', 'platform_machine': 'arm64'}` → `[True, False, True, True]`

In [ ]:
def markers(marker_list, env):
    """
    Markers decide whether a requirement applies here. Support comparisons of a
    variable with a quoted value, joined by and/or (and binds tighter). Version
    variables (names ending in version) compare as versions. Return a bool per
    marker for the environment env.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(markers, [
    ((["python_version >= '3.10'", "python_version < '3.11' and sys_platform == 'linux'", "sys_platform == 'win32' or platform_machine == 'arm64'", "python_version >= '3.9' and python_version < '3.10' or sys_platform == 'darwin'"], {'python_version': '3.12', 'sys_platform': 'darwin', 'platform_machine': 'arm64'}), [True, False, True, True]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Split on or/and and evaluate each comparison by hand.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

The same with the `marker_true` helper.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Split on or/and and evaluate each comparison by hand.

```python
def vkey(v):
    m = re.fullmatch(r"(\d+(?:\.\d+)*)(?:(a|b|rc)(\d+))?", v)
    release = [int(x) for x in m.group(1).split(".")]
    release += [0] * (4 - len(release))
    pre = (PRE[m.group(2)], int(m.group(3))) if m.group(2) else (3, 0)
    return (tuple(release), pre)

def marker_true(marker, env):
    def atom(text):
        m = re.fullmatch(r"\s*(\w+)\s*(==|!=|>=|<=|>|<)\s*'([^']*)'\s*", text)
        var, op, val = m.groups()
        left = env[var]
        if var.endswith("version"):
            a, b = vkey(left), vkey(val)
        else:
            a, b = left, val
        return {"==": a == b, "!=": a != b, ">=": a >= b, "<=": a <= b, ">": a > b, "<": a < b}[op]
    return any(all(atom(t) for t in part.split(" and ")) for part in marker.split(" or "))


def markers(marker_list, env):
    out = []
    for m in marker_list:
        result = False
        for part in m.split(" or "):
            ok = True
            for t in part.split(" and "):
                if not marker_true(t, env):
                    ok = False
            if ok:
                result = True
        out.append(result)
    return out
```

**Cost:** O(m · t) time, O(t) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** The same with the `marker_true` helper.

```python
def vkey(v):
    m = re.fullmatch(r"(\d+(?:\.\d+)*)(?:(a|b|rc)(\d+))?", v)
    release = [int(x) for x in m.group(1).split(".")]
    release += [0] * (4 - len(release))
    pre = (PRE[m.group(2)], int(m.group(3))) if m.group(2) else (3, 0)
    return (tuple(release), pre)

def marker_true(marker, env):
    def atom(text):
        m = re.fullmatch(r"\s*(\w+)\s*(==|!=|>=|<=|>|<)\s*'([^']*)'\s*", text)
        var, op, val = m.groups()
        left = env[var]
        if var.endswith("version"):
            a, b = vkey(left), vkey(val)
        else:
            a, b = left, val
        return {"==": a == b, "!=": a != b, ">=": a >= b, "<=": a <= b, ">": a > b, "<": a < b}[op]
    return any(all(atom(t) for t in part.split(" and ")) for part in marker.split(" or "))


def markers(marker_list, env):
    return [marker_true(m, env) for m in marker_list]
```

**Cost:** O(m · t) time, O(t) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Split on or/and and evaluate each comparison by hand. That costs O(m · t) time and O(t) space."

"I can do better: The same with the `marker_true` helper. That's O(m · t) time and O(t) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/engineering-toolbox/#/q/packaging/evaluate-environment-markers)

<a id="q11"></a>
### Q11. Read a Wheel Filename

🟡 Interview standard · *Will this wheel install here?*

Wheel names are `name-version(-build)?-python-abi-platform.whl`, and each tag part may hold several tags joined by `.`. For each file, return `(name, version, compatible)` where compatible means some combination of its python, abi and platform tags is in `supported` (a list of `'py-abi-platform'` strings).

**Examples**

- `files = ['numpy-2.1.3-cp312-cp312-macosx_14_0_arm64.whl', 'requests-2.32.3-py3-none-any.whl', 'pydantic_core-2.23.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl', 'six-1.16.0-py2.py3-none-any.whl'], supported = ['cp312-cp312-macosx_14_0_arm64', 'cp312-none-any', 'py3-none-any']` → `[('numpy', '2.1.3', True), ('requests', '2.32.3', True), ('pydantic_core', '2.23.4', False), ('six', '1.16.0', True)]`

In [ ]:
def wheel_info(files, supported):
    """
    Wheel names are name-version(-build)?-python-abi-platform.whl, and each tag
    part may hold several tags joined by .. For each file, return (name,
    version, compatible) where compatible means some combination of its python,
    abi and platform tags is in supported (a list of 'py-abi-platform' strings).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(wheel_info, [
    ((['numpy-2.1.3-cp312-cp312-macosx_14_0_arm64.whl', 'requests-2.32.3-py3-none-any.whl', 'pydantic_core-2.23.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl', 'six-1.16.0-py2.py3-none-any.whl'], ['cp312-cp312-macosx_14_0_arm64', 'cp312-none-any', 'py3-none-any']), [('numpy', '2.1.3', True), ('requests', '2.32.3', True), ('pydantic_core', '2.23.4', False), ('six', '1.16.0', True)]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Split and check every tag combination with nested loops.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`itertools.product` of the tag sets.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Split and check every tag combination with nested loops.

```python
def wheel_info(files, supported):
    out = []
    for f in files:
        parts = f[:-4].split("-")
        name, version = parts[0], parts[1]
        py, abi, plat = parts[-3], parts[-2], parts[-1]
        ok = False
        for a in py.split("."):
            for b in abi.split("."):
                for c in plat.split("."):
                    if f"{a}-{b}-{c}" in supported:
                        ok = True
        out.append((name, version, ok))
    return out
```

**Cost:** O(f · t³) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** `itertools.product` of the tag sets.

```python
def wheel_info(files, supported):
    sup = set(supported)
    out = []
    for f in files:
        p = f.removesuffix(".whl").split("-")
        tags = itertools.product(*(x.split(".") for x in p[-3:]))
        out.append((p[0], p[1], any("-".join(t) in sup for t in tags)))
    return out
```

**Cost:** O(f · t³) time, O(s) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Split and check every tag combination with nested loops. That costs O(f · t³) time and O(1) space."

"I can do better: `itertools.product` of the tag sets. That's O(f · t³) time and O(s) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/engineering-toolbox/#/q/packaging/read-a-wheel-filename)

<a id="q12"></a>
### Q12. Find Vulnerable Packages

🟡 Interview standard · *pip-audit*

Advisories are `(id, package, affected spec, fixed version)`. Return `(package, installed version, advisory id, fixed version)` for every installed package matching an advisory, sorted.

**Examples**

- `installed = {'jinja2': '3.1.3', 'requests': '2.32.3', 'urllib3': '1.26.18', 'cryptography': '41.0.7', 'django': '5.1.3'}, advisories = [('GHSA-h75v', 'jinja2', '<3.1.4', '3.1.4'), ('GHSA-9wx4', 'requests', '<2.32.0', '2.32.0'), ('GHSA-34jh', 'urllib3', '>=1.26.0,<1.26.19', '1.26.19'), ('GHSA-3ww4', 'cryptography', '>=37.0.0,<42.0.4', '42.0.4'), ('GHSA-x2fm', 'django', '>=5.0,<5.0.8', '5.0.8')]` → `[('cryptography', '41.0.7', 'GHSA-3ww4', '42.0.4'), ('jinja2', '3.1.3', 'GHSA-h75v', '3.1.4'), ('urllib3', '1.26.18', 'GHSA-34jh', '1.26.19')]`

In [ ]:
def vulnerable(installed, advisories):
    """
    Advisories are (id, package, affected spec, fixed version). Return (package,
    installed version, advisory id, fixed version) for every installed package
    matching an advisory, sorted.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(vulnerable, [
    (({'jinja2': '3.1.3', 'requests': '2.32.3', 'urllib3': '1.26.18', 'cryptography': '41.0.7', 'django': '5.1.3'}, [('GHSA-h75v', 'jinja2', '<3.1.4', '3.1.4'), ('GHSA-9wx4', 'requests', '<2.32.0', '2.32.0'), ('GHSA-34jh', 'urllib3', '>=1.26.0,<1.26.19', '1.26.19'), ('GHSA-3ww4', 'cryptography', '>=37.0.0,<42.0.4', '42.0.4'), ('GHSA-x2fm', 'django', '>=5.0,<5.0.8', '5.0.8')]), [('cryptography', '41.0.7', 'GHSA-3ww4', '42.0.4'), ('jinja2', '3.1.3', 'GHSA-h75v', '3.1.4'), ('urllib3', '1.26.18', 'GHSA-34jh', '1.26.19')]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Loop advisories and check installed versions.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A sorted comprehension.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Loop advisories and check installed versions.

```python
def vkey(v):
    m = re.fullmatch(r"(\d+(?:\.\d+)*)(?:(a|b|rc)(\d+))?", v)
    release = [int(x) for x in m.group(1).split(".")]
    release += [0] * (4 - len(release))
    pre = (PRE[m.group(2)], int(m.group(3))) if m.group(2) else (3, 0)
    return (tuple(release), pre)

def satisfies(v, spec):
    for part in [p.strip() for p in spec.split(",") if p.strip()]:
        op, target = re.fullmatch(r"(~=|==|!=|>=|<=|>|<)\s*(\S+)", part).groups()
        if target.endswith(".*"):
            prefix = target[:-2].split(".")
            same = [str(x) for x in vkey(v)[0][:len(prefix)]] == prefix
            if (op == "==") != same:
                return False
            continue
        a, b = vkey(v), vkey(target)
        if op == "~=":
            head = target.split(".")[:-1]
            if not (a >= b and [str(x) for x in a[0][:len(head)]] == head):
                return False
        elif not {"==": a == b, "!=": a != b, ">=": a >= b, "<=": a <= b, ">": a > b, "<": a < b}[op]:
            return False
    return True


def vulnerable(installed, advisories):
    out = []
    for aid, pkg, spec, fixed in advisories:
        if pkg in installed:
            if satisfies(installed[pkg], spec):
                out.append((pkg, installed[pkg], aid, fixed))
    out.sort()
    return out
```

**Cost:** O(a) time, O(a) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A sorted comprehension.

```python
def vkey(v):
    m = re.fullmatch(r"(\d+(?:\.\d+)*)(?:(a|b|rc)(\d+))?", v)
    release = [int(x) for x in m.group(1).split(".")]
    release += [0] * (4 - len(release))
    pre = (PRE[m.group(2)], int(m.group(3))) if m.group(2) else (3, 0)
    return (tuple(release), pre)

def satisfies(v, spec):
    for part in [p.strip() for p in spec.split(",") if p.strip()]:
        op, target = re.fullmatch(r"(~=|==|!=|>=|<=|>|<)\s*(\S+)", part).groups()
        if target.endswith(".*"):
            prefix = target[:-2].split(".")
            same = [str(x) for x in vkey(v)[0][:len(prefix)]] == prefix
            if (op == "==") != same:
                return False
            continue
        a, b = vkey(v), vkey(target)
        if op == "~=":
            head = target.split(".")[:-1]
            if not (a >= b and [str(x) for x in a[0][:len(head)]] == head):
                return False
        elif not {"==": a == b, "!=": a != b, ">=": a >= b, "<=": a <= b, ">": a > b, "<": a < b}[op]:
            return False
    return True


def vulnerable(installed, advisories):
    return sorted((p, installed[p], a, f) for a, p, s, f in advisories if p in installed and satisfies(installed[p], s))
```

**Cost:** O(a log a) time, O(a) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Loop advisories and check installed versions. That costs O(a) time and O(a) space."

"I can do better: A sorted comprehension. That's O(a log a) time and O(a) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/engineering-toolbox/#/q/packaging/find-vulnerable-packages)

<a id="q13"></a>
### Q13. Measure the Dependency Tree

🟡 Interview standard · *uv tree*

`deps` maps each package to its direct dependencies. For each top-level package, return `(number of distinct transitive dependencies, depth of the deepest chain)` (a package with no deps has depth 0).

**Examples**

- `deps = {'fastapi': ['starlette', 'pydantic'], 'starlette': ['anyio'], 'anyio': ['idna', 'sniffio'], 'pydantic': ['pydantic-core', 'annotated-types', 'typing-extensions'], 'pydantic-core': ['typing-extensions'], 'annotated-types': [], 'typing-extensions': [], 'idna': [], 'sniffio': [], 'requests': ['urllib3', 'idna', 'certifi', 'charset-normalizer'], 'urllib3': [], 'certifi': [], 'charset-normalizer': []}, top = ['fastapi', 'requests']` → `{'fastapi': (8, 3), 'requests': (4, 1)}`

In [ ]:
def tree_stats(deps, top):
    """
    deps maps each package to its direct dependencies. For each top-level
    package, return (number of distinct transitive dependencies, depth of the
    deepest chain) (a package with no deps has depth 0).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(tree_stats, [
    (({'fastapi': ['starlette', 'pydantic'], 'starlette': ['anyio'], 'anyio': ['idna', 'sniffio'], 'pydantic': ['pydantic-core', 'annotated-types', 'typing-extensions'], 'pydantic-core': ['typing-extensions'], 'annotated-types': [], 'typing-extensions': [], 'idna': [], 'sniffio': [], 'requests': ['urllib3', 'idna', 'certifi', 'charset-normalizer'], 'urllib3': [], 'certifi': [], 'charset-normalizer': []}, ['fastapi', 'requests']), {'fastapi': (8, 3), 'requests': (4, 1)}),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Recursive DFS for each question separately.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Memoised depth and descendant sets.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Recursive DFS for each question separately.

```python
def tree_stats(deps, top):
    out = {}
    for t in top:
        seen, stack = set(), list(deps[t])
        while stack:
            d = stack.pop()
            if d not in seen:
                seen.add(d)
                stack += deps[d]
        def depth(p):
            return 0 if not deps[p] else 1 + max(depth(c) for c in deps[p])
        out[t] = (len(seen), depth(t))
    return out
```

**Cost:** O(t · (n + e)) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Memoised depth and descendant sets.

```python
def tree_stats(deps, top):
    from functools import lru_cache
    @lru_cache(None)
    def below(p):
        s = set(deps[p])
        for c in deps[p]:
            s |= below(c)
        return frozenset(s)
    @lru_cache(None)
    def depth(p):
        return 1 + max(map(depth, deps[p])) if deps[p] else 0
    return {t: (len(below(t)), depth(t)) for t in top}
```

**Cost:** O(n + e) time, O(n²) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Recursive DFS for each question separately. That costs O(t · (n + e)) time and O(n) space."

"I can do better: Memoised depth and descendant sets. That's O(n + e) time and O(n²) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `deps = {'a': ['b'], 'b': []}, top = ['a']`

1. below() returns frozenset()
2. below() returns frozenset({'b'})
3. depth() returns 0
4. depth() returns 1
5. tree\_stats() returns {'a': (1, 1)}

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/engineering-toolbox/#/q/packaging/measure-the-dependency-tree)

<a id="q14"></a>
### Q14. Why Is This Package Installed?

🟡 Interview standard · *Reverse dependencies*

Return, for each target, the top-level packages that pull it in (directly or indirectly), sorted. Use the same graph shape as the previous question; `top` are the packages you asked for.

**Examples**

- `deps = {'fastapi': ['starlette', 'pydantic'], 'starlette': ['anyio'], 'anyio': ['idna', 'sniffio'], 'pydantic': ['typing-extensions'], 'typing-extensions': [], 'idna': [], 'sniffio': [], 'requests': ['urllib3', 'idna', 'certifi'], 'urllib3': [], 'certifi': [], 'httpx': ['anyio', 'certifi', 'idna']}, top = ['fastapi', 'requests', 'httpx'], targets = ['idna', 'certifi', 'typing-extensions']` → `{'idna': ['fastapi', 'httpx', 'requests'], 'certifi': ['httpx', 'requests'], 'typing-extensions': ['fastapi']}`

In [ ]:
from collections import defaultdict, deque

def why_installed(deps, top, targets):
    """
    Return, for each target, the top-level packages that pull it in (directly or
    indirectly), sorted. Use the same graph shape as the previous question; top
    are the packages you asked for.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(why_installed, [
    (({'fastapi': ['starlette', 'pydantic'], 'starlette': ['anyio'], 'anyio': ['idna', 'sniffio'], 'pydantic': ['typing-extensions'], 'typing-extensions': [], 'idna': [], 'sniffio': [], 'requests': ['urllib3', 'idna', 'certifi'], 'urllib3': [], 'certifi': [], 'httpx': ['anyio', 'certifi', 'idna']}, ['fastapi', 'requests', 'httpx'], ['idna', 'certifi', 'typing-extensions']), {'idna': ['fastapi', 'httpx', 'requests'], 'certifi': ['httpx', 'requests'], 'typing-extensions': ['fastapi']}),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

DFS from each top-level package.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Reverse the graph and walk up from each target.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** DFS from each top-level package.

```python
def why_installed(deps, top, targets):
    reach = {}
    for t in top:
        seen, stack = set(), [t]
        while stack:
            d = stack.pop()
            if d not in seen:
                seen.add(d)
                stack += deps[d]
        reach[t] = seen
    return {x: sorted(t for t in top if x in reach[t] and x != t) for x in targets}
```

**Cost:** O(t · (n + e)) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Reverse the graph and walk up from each target.

```python
def why_installed(deps, top, targets):
    up = defaultdict(list)
    for p, ds in deps.items():
        for d in ds:
            up[d].append(p)
    out = {}
    for x in targets:
        seen, q = set(), deque([x])
        while q:
            for parent in up[q.popleft()]:
                if parent not in seen:
                    seen.add(parent)
                    q.append(parent)
        out[x] = sorted(seen & set(top))
    return out
```

**Cost:** O(q · (n + e)) time, O(n + e) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: DFS from each top-level package. That costs O(t · (n + e)) time and O(n) space."

"I can do better: Reverse the graph and walk up from each target. That's O(q · (n + e)) time and O(n + e) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/engineering-toolbox/#/q/packaging/why-is-this-package-installed)

<a id="q15"></a>
### Q15. Spot Version Conflicts

🟡 Interview standard · *Two packages want different things*

Several packages constrain the same dependency. For each dependency in `constraints` (dep → list of `(who, spec)`), return the versions from `available[dep]` that satisfy **all** constraints, or the string `'conflict: '` followed by the requesters sorted and comma-separated if none do.

**Examples**

- `constraints = {'pydantic': [('fastapi', '>=1.7.4,!=2.0.0'), ('langchain', '>=2,<3'), ('old-sdk', '<2')], 'httpx': [('openai', '>=0.23,<1'), ('anthropic', '>=0.25,<1')]}, available = {'pydantic': ['1.10.18', '2.0.0', '2.9.2'], 'httpx': ['0.24.1', '0.27.2', '0.28.0']}` → `{'pydantic': 'conflict: fastapi, langchain, old-sdk', 'httpx': ['0.27.2', '0.28.0']}`

In [ ]:
def conflicts(constraints, available):
    """
    Several packages constrain the same dependency. For each dependency in
    constraints (dep → list of (who, spec)), return the versions from
    available[dep] that satisfy all constraints, or the string 'conflict: '
    followed by the requesters sorted and comma-separated if none do.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(conflicts, [
    (({'pydantic': [('fastapi', '>=1.7.4,!=2.0.0'), ('langchain', '>=2,<3'), ('old-sdk', '<2')], 'httpx': [('openai', '>=0.23,<1'), ('anthropic', '>=0.25,<1')]}, {'pydantic': ['1.10.18', '2.0.0', '2.9.2'], 'httpx': ['0.24.1', '0.27.2', '0.28.0']}), {'pydantic': 'conflict: fastapi, langchain, old-sdk', 'httpx': ['0.27.2', '0.28.0']}),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Check every version against every constraint.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Combine constraints into one spec and filter.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Check every version against every constraint.

```python
def vkey(v):
    m = re.fullmatch(r"(\d+(?:\.\d+)*)(?:(a|b|rc)(\d+))?", v)
    release = [int(x) for x in m.group(1).split(".")]
    release += [0] * (4 - len(release))
    pre = (PRE[m.group(2)], int(m.group(3))) if m.group(2) else (3, 0)
    return (tuple(release), pre)

def satisfies(v, spec):
    for part in [p.strip() for p in spec.split(",") if p.strip()]:
        op, target = re.fullmatch(r"(~=|==|!=|>=|<=|>|<)\s*(\S+)", part).groups()
        if target.endswith(".*"):
            prefix = target[:-2].split(".")
            same = [str(x) for x in vkey(v)[0][:len(prefix)]] == prefix
            if (op == "==") != same:
                return False
            continue
        a, b = vkey(v), vkey(target)
        if op == "~=":
            head = target.split(".")[:-1]
            if not (a >= b and [str(x) for x in a[0][:len(head)]] == head):
                return False
        elif not {"==": a == b, "!=": a != b, ">=": a >= b, "<=": a <= b, ">": a > b, "<": a < b}[op]:
            return False
    return True


def conflicts(constraints, available):
    out = {}
    for dep, reqs in constraints.items():
        ok = []
        for v in available[dep]:
            good = True
            for _, spec in reqs:
                if not satisfies(v, spec):
                    good = False
            if good:
                ok.append(v)
        out[dep] = ok if ok else "conflict: " + ", ".join(sorted(w for w, _ in reqs))
    return out
```

**Cost:** O(d · v · c) time, O(v) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Combine constraints into one spec and filter.

```python
def vkey(v):
    m = re.fullmatch(r"(\d+(?:\.\d+)*)(?:(a|b|rc)(\d+))?", v)
    release = [int(x) for x in m.group(1).split(".")]
    release += [0] * (4 - len(release))
    pre = (PRE[m.group(2)], int(m.group(3))) if m.group(2) else (3, 0)
    return (tuple(release), pre)

def satisfies(v, spec):
    for part in [p.strip() for p in spec.split(",") if p.strip()]:
        op, target = re.fullmatch(r"(~=|==|!=|>=|<=|>|<)\s*(\S+)", part).groups()
        if target.endswith(".*"):
            prefix = target[:-2].split(".")
            same = [str(x) for x in vkey(v)[0][:len(prefix)]] == prefix
            if (op == "==") != same:
                return False
            continue
        a, b = vkey(v), vkey(target)
        if op == "~=":
            head = target.split(".")[:-1]
            if not (a >= b and [str(x) for x in a[0][:len(head)]] == head):
                return False
        elif not {"==": a == b, "!=": a != b, ">=": a >= b, "<=": a <= b, ">": a > b, "<": a < b}[op]:
            return False
    return True


def conflicts(constraints, available):
    out = {}
    for dep, reqs in constraints.items():
        combined = ",".join(s for _, s in reqs)
        ok = [v for v in available[dep] if satisfies(v, combined)]
        out[dep] = ok or "conflict: " + ", ".join(sorted(w for w, _ in reqs))
    return out
```

**Cost:** O(d · v · c) time, O(v) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Check every version against every constraint. That costs O(d · v · c) time and O(v) space."

"I can do better: Combine constraints into one spec and filter. That's O(d · v · c) time and O(v) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/engineering-toolbox/#/q/packaging/spot-version-conflicts)

<a id="q16"></a>
### Q16. Configuration Precedence

🟡 Interview standard · *Defaults &lt; file &lt; env &lt; flags*

Settings come from four layers, later ones winning: `defaults`, the config `file`, environment variables (`APP_` prefix, lower-cased key: `APP_DB_URL` → `db_url`; other variables ignored) and command-line `flags`. Return `{key: [value, source]}` sorted by key.

**Examples**

- `defaults = {'db_url': 'sqlite:///dev.db', 'log_level': 'INFO', 'workers': '1', 'debug': 'false'}, file = {'log_level': 'WARNING', 'workers': '4'}, environ = {'APP_DB_URL': 'postgres://prod/db', 'APP_WORKERS': '8', 'HOME': '/home/asha'}, flags = {'debug': 'true'}` → `{'db_url': ['postgres://prod/db', 'env'], 'debug': ['true', 'flag'], 'log_level': ['WARNING', 'file'], 'workers': ['8', 'env']}`

In [ ]:
def resolve_config(defaults, file, environ, flags):
    """
    Settings come from four layers, later ones winning: defaults, the config
    file, environment variables (APP_ prefix, lower-cased key: APP_DB_URL →
    db_url; other variables ignored) and command-line flags. Return {key:
    [value, source]} sorted by key.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(resolve_config, [
    (({'db_url': 'sqlite:///dev.db', 'log_level': 'INFO', 'workers': '1', 'debug': 'false'}, {'log_level': 'WARNING', 'workers': '4'}, {'APP_DB_URL': 'postgres://prod/db', 'APP_WORKERS': '8', 'HOME': '/home/asha'}, {'debug': 'true'}), {'db_url': ['postgres://prod/db', 'env'], 'debug': ['true', 'flag'], 'log_level': ['WARNING', 'file'], 'workers': ['8', 'env']}),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Apply each layer in a loop, remembering the source.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Merge (value, source) dicts in order with dict unions.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Apply each layer in a loop, remembering the source.

```python
def resolve_config(defaults, file, environ, flags):
    out = {}
    for k, v in defaults.items():
        out[k] = [v, "default"]
    for k, v in file.items():
        out[k] = [v, "file"]
    for k, v in environ.items():
        if k.startswith("APP_"):
            out[k[4:].lower()] = [v, "env"]
    for k, v in flags.items():
        out[k] = [v, "flag"]
    return dict(sorted(out.items()))
```

**Cost:** O(k) time, O(k) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Merge (value, source) dicts in order with dict unions.

```python
def resolve_config(defaults, file, environ, flags):
    env = {k[4:].lower(): v for k, v in environ.items() if k.startswith("APP_")}
    layers = [(defaults, "default"), (file, "file"), (env, "env"), (flags, "flag")]
    merged = {}
    for layer, src in layers:
        merged |= {k: [v, src] for k, v in layer.items()}
    return dict(sorted(merged.items()))
```

**Cost:** O(k) time, O(k) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Apply each layer in a loop, remembering the source. That costs O(k) time and O(k) space."

"I can do better: Merge (value, source) dicts in order with dict unions. That's O(k) time and O(k) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/engineering-toolbox/#/q/packaging/configuration-precedence)

<a id="q17"></a>
### Q17. Parse a .env File

🟡 Interview standard · *Twelve-factor config*

Parse `.env` text into a dict: skip blank lines and `#` comments; allow an optional `export ` prefix; strip matching single or double quotes from values; drop an unquoted value's trailing ` # comment`; and expand `${NAME}` using variables defined earlier in the file (unknown → empty).

**Examples**

- `text = '# database\nexport DB_HOST=localhost\nDB_PORT=5432\nDB_USER=\'app\'\nDB_URL="postgres://${DB_USER}@${DB_HOST}:${DB_PORT}/shop"\nDEBUG=true # local only\n\nGREETING="hello # not a comment"\nMISSING=${NOPE}x\n'` → `{'DB_HOST': 'localhost', 'DB_PORT': '5432', 'DB_USER': 'app', 'DB_URL': 'postgres://app@localhost:5432/shop', 'DEBUG': 'true', 'GREETING': 'hello # not a comment', 'MISSING': 'x'}`

In [ ]:
def parse_env(text):
    """
    Parse .env text into a dict: skip blank lines and # comments; allow an
    optional export  prefix; strip matching single or double quotes from values;
    drop an unquoted value's trailing  # comment; and expand ${NAME} using
    variables defined earlier in the file (unknown → empty).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(parse_env, [
    (('# database\nexport DB_HOST=localhost\nDB_PORT=5432\nDB_USER=\'app\'\nDB_URL="postgres://${DB_USER}@${DB_HOST}:${DB_PORT}/shop"\nDEBUG=true # local only\n\nGREETING="hello # not a comment"\nMISSING=${NOPE}x\n',), {'DB_HOST': 'localhost', 'DB_PORT': '5432', 'DB_USER': 'app', 'DB_URL': 'postgres://app@localhost:5432/shop', 'DEBUG': 'true', 'GREETING': 'hello # not a comment', 'MISSING': 'x'}),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Line by line with string methods.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A regex per line and re.sub for expansion.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Line by line with string methods.

```python
def parse_env(text):
    env = {}
    for raw in text.splitlines():
        line = raw.strip()
        if not line or line.startswith("#"):
            continue
        if line.startswith("export "):
            line = line[7:]
        key, _, value = line.partition("=")
        value = value.strip()
        if len(value) >= 2 and value[0] == value[-1] and value[0] in "'\"":
            value = value[1:-1]
        elif " #" in value:
            value = value.split(" #")[0].strip()
        while "${" in value:
            start = value.index("${")
            end = value.index("}", start)
            value = value[:start] + env.get(value[start + 2:end], "") + value[end + 1:]
        env[key.strip()] = value
    return env
```

**Cost:** O(n · L) time, O(k) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A regex per line and re.sub for expansion.

```python
def parse_env(text):
    env = {}
    for line in text.splitlines():
        m = re.match(r"^\s*(?:export\s+)?([A-Za-z_]\w*)\s*=\s*(.*?)\s*$", line)
        if not m or line.strip().startswith("#"):
            continue
        key, v = m.groups()
        q = re.fullmatch(r"(['\"])(.*)\1", v)
        v = q.group(2) if q else re.sub(r"\s+#.*$", "", v)
        env[key] = re.sub(r"\$\{(\w+)\}", lambda g: env.get(g.group(1), ""), v)
    return env
```

**Cost:** O(n · L) time, O(k) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Line by line with string methods. That costs O(n · L) time and O(k) space."

"I can do better: A regex per line and re.sub for expansion. That's O(n · L) time and O(k) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/engineering-toolbox/#/q/packaging/parse-a-env-file)

<a id="q18"></a>
### Q18. Safe Automatic Upgrades

🟡 Interview standard · *Dependabot and Renovate policies*

For each installed package, pick the newest available non-pre-release upgrade allowed by the policy: `'patch'` (same major and minor), `'minor'` (same major) or `'major'` (anything newer). Return `{package: new version or None}`.

**Examples**

- `installed = {'django': '5.0.4', 'requests': '2.31.0', 'numpy': '1.26.2'}, available = {'django': ['5.0.4', '5.0.9', '5.1.3', '6.0a1'], 'requests': ['2.31.0', '2.32.3'], 'numpy': ['1.26.2', '1.26.4', '2.1.3']}, policy = 'patch'` → `{'django': '5.0.9', 'requests': None, 'numpy': '1.26.4'}`
- `installed = {'django': '5.0.4', 'requests': '2.31.0', 'numpy': '1.26.2'}, available = {'django': ['5.0.4', '5.0.9', '5.1.3', '6.0a1'], 'requests': ['2.31.0', '2.32.3'], 'numpy': ['1.26.2', '1.26.4', '2.1.3']}, policy = 'minor'` → `{'django': '5.1.3', 'requests': '2.32.3', 'numpy': '1.26.4'}`

In [ ]:
def safe_upgrade(installed, available, policy):
    """
    For each installed package, pick the newest available non-pre-release
    upgrade allowed by the policy: 'patch' (same major and minor), 'minor' (same
    major) or 'major' (anything newer). Return {package: new version or None}.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(safe_upgrade, [
    (({'django': '5.0.4', 'requests': '2.31.0', 'numpy': '1.26.2'}, {'django': ['5.0.4', '5.0.9', '5.1.3', '6.0a1'], 'requests': ['2.31.0', '2.32.3'], 'numpy': ['1.26.2', '1.26.4', '2.1.3']}, 'patch'), {'django': '5.0.9', 'requests': None, 'numpy': '1.26.4'}),
    (({'django': '5.0.4', 'requests': '2.31.0', 'numpy': '1.26.2'}, {'django': ['5.0.4', '5.0.9', '5.1.3', '6.0a1'], 'requests': ['2.31.0', '2.32.3'], 'numpy': ['1.26.2', '1.26.4', '2.1.3']}, 'minor'), {'django': '5.1.3', 'requests': '2.32.3', 'numpy': '1.26.4'}),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Check each candidate with if-statements.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Compare release prefixes and take the max.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Check each candidate with if-statements.

```python
def vkey(v):
    m = re.fullmatch(r"(\d+(?:\.\d+)*)(?:(a|b|rc)(\d+))?", v)
    release = [int(x) for x in m.group(1).split(".")]
    release += [0] * (4 - len(release))
    pre = (PRE[m.group(2)], int(m.group(3))) if m.group(2) else (3, 0)
    return (tuple(release), pre)

def is_pre(v):
    return vkey(v)[1][0] < 3


def safe_upgrade(installed, available, policy):
    out = {}
    for p, cur in installed.items():
        c = vkey(cur)[0]
        best = None
        for v in available[p]:
            if is_pre(v) or vkey(v) <= vkey(cur):
                continue
            n = vkey(v)[0]
            if policy == "patch" and n[:2] != c[:2]:
                continue
            if policy == "minor" and n[0] != c[0]:
                continue
            if best is None or vkey(v) > vkey(best):
                best = v
        out[p] = best
    return out
```

**Cost:** O(p · v) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Compare release prefixes and take the max.

```python
def vkey(v):
    m = re.fullmatch(r"(\d+(?:\.\d+)*)(?:(a|b|rc)(\d+))?", v)
    release = [int(x) for x in m.group(1).split(".")]
    release += [0] * (4 - len(release))
    pre = (PRE[m.group(2)], int(m.group(3))) if m.group(2) else (3, 0)
    return (tuple(release), pre)

def is_pre(v):
    return vkey(v)[1][0] < 3


def safe_upgrade(installed, available, policy):
    keep = {"patch": 2, "minor": 1, "major": 0}[policy]
    def allowed(cur, v):
        return not is_pre(v) and vkey(v) > vkey(cur) and vkey(v)[0][:keep] == vkey(cur)[0][:keep]
    return {p: max((v for v in available[p] if allowed(cur, v)), key=vkey, default=None) for p, cur in installed.items()}
```

**Cost:** O(p · v) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Check each candidate with if-statements. That costs O(p · v) time and O(1) space."

"I can do better: Compare release prefixes and take the max. That's O(p · v) time and O(1) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `installed = {'a': '1.0.0'}, available = {'a': ['1.0.1', '1.1.0']}, policy = 'patch'`

1. allowed() returns True
2. allowed() returns False
3. safe\_upgrade() returns {'a': '1.0.1'}

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/engineering-toolbox/#/q/packaging/safe-automatic-upgrades)

<a id="q19"></a>
### Q19. Is This Release Breaking?

🟡 Interview standard · *Semantic versioning for libraries*

Compare a library's public API before and after (function → list of parameters; a parameter ending in `=` is optional). Return the required bump: `'major'` if a function or required-or-optional parameter was removed, or a new **required** parameter was added; `'minor'` if a function or optional parameter was added; otherwise `'patch'`.

**Examples**

- `old = {'connect': ['url', 'timeout='], 'fetch': ['key']}, new = {'connect': ['url', 'timeout=', 'retries='], 'fetch': ['key'], 'close': []}` → `'minor'`
- `old = {'connect': ['url', 'timeout='], 'fetch': ['key']}, new = {'connect': ['url', 'timeout='], 'fetch': ['key', 'region']}` → `'major'`
- `old = {'connect': ['url', 'timeout='], 'fetch': ['key']}, new = {'connect': ['url']}` → `'major'`

In [ ]:
def required_bump(old, new):
    """
    Compare a library's public API before and after (function → list of
    parameters; a parameter ending in = is optional). Return the required bump:
    'major' if a function or required-or-optional parameter was removed, or a
    new required parameter was added; 'minor' if a function or optional
    parameter was added; otherwise 'patch'.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(required_bump, [
    (({'connect': ['url', 'timeout='], 'fetch': ['key']}, {'connect': ['url', 'timeout=', 'retries='], 'fetch': ['key'], 'close': []}), 'minor'),
    (({'connect': ['url', 'timeout='], 'fetch': ['key']}, {'connect': ['url', 'timeout='], 'fetch': ['key', 'region']}), 'major'),
    (({'connect': ['url', 'timeout='], 'fetch': ['key']}, {'connect': ['url']}), 'major'),
    (({'connect': ['url']}, {'connect': ['url']}), 'patch'),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Flags set while looping over functions and parameters.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Set differences.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Flags set while looping over functions and parameters.

```python
def required_bump(old, new):
    major = minor = False
    for fn, params in old.items():
        if fn not in new:
            major = True
            continue
        for p in params:
            if p not in new[fn]:
                major = True
        for p in new[fn]:
            if p not in params:
                if p.endswith("="):
                    minor = True
                else:
                    major = True
    for fn in new:
        if fn not in old:
            minor = True
    return "major" if major else "minor" if minor else "patch"
```

**Cost:** O(f · p) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Set differences.

```python
def required_bump(old, new):
    removed = set(old) - set(new) or any(set(old[f]) - set(new[f]) for f in old if f in new)
    added = {p for f in old if f in new for p in set(new[f]) - set(old[f])}
    if removed or any(not p.endswith("=") for p in added):
        return "major"
    return "minor" if added or set(new) - set(old) else "patch"
```

**Cost:** O(f · p) time, O(p) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Flags set while looping over functions and parameters. That costs O(f · p) time and O(1) space."

"I can do better: Set differences. That's O(f · p) time and O(p) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/engineering-toolbox/#/q/packaging/is-this-release-breaking)

<a id="q20"></a>
### Q20. Resolve Dependencies

🔴 Stretch · *Backtracking like pip and uv*

`index[pkg][version]` lists that version's requirements `{dep: spec}`; every package in the index is part of the install. Choose one version per package so that the root requirements and all requirements between chosen versions hold, preferring newer versions for packages earlier in alphabetical order. Return the choice (sorted dict) or `'conflict'`.

**Examples**

- `index = {'app-lib': {'2.0.0': {'httpx': '>=0.27', 'pydantic': '>=2'}, '1.5.0': {'httpx': '>=0.24', 'pydantic': '>=1.10,<2'}}, 'httpx': {'0.28.0': {'anyio': '>=4'}, '0.27.0': {'anyio': '>=3'}, '0.24.0': {'anyio': '>=3,<4'}}, 'anyio': {'4.4.0': {}, '3.7.1': {}}, 'pydantic': {'2.9.0': {}, '1.10.18': {}}}, root = {'pydantic': '<2'}` → `{'anyio': '4.4.0', 'app-lib': '1.5.0', 'httpx': '0.28.0', 'pydantic': '1.10.18'}`
- `index = {'a': {'2.0': {'b': '>=2'}}, 'b': {'1.0': {}}}, root = {}` → `'conflict'`

In [ ]:
def resolve(index, root):
    """
    index[pkg][version] lists that version's requirements {dep: spec}; every
    package in the index is part of the install. Choose one version per package
    so that the root requirements and all requirements between chosen versions
    hold, preferring newer versions for packages earlier in alphabetical order.
    Return the choice (sorted dict) or 'conflict'.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(resolve, [
    (({'app-lib': {'2.0.0': {'httpx': '>=0.27', 'pydantic': '>=2'}, '1.5.0': {'httpx': '>=0.24', 'pydantic': '>=1.10,<2'}}, 'httpx': {'0.28.0': {'anyio': '>=4'}, '0.27.0': {'anyio': '>=3'}, '0.24.0': {'anyio': '>=3,<4'}}, 'anyio': {'4.4.0': {}, '3.7.1': {}}, 'pydantic': {'2.9.0': {}, '1.10.18': {}}}, {'pydantic': '<2'}), {'anyio': '4.4.0', 'app-lib': '1.5.0', 'httpx': '0.28.0', 'pydantic': '1.10.18'}),
    (({'a': {'2.0': {'b': '>=2'}}, 'b': {'1.0': {}}}, {}), 'conflict'),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Try every combination, newest first, and keep the first that works.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Backtracking: choose package by package, undo when a constraint breaks.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Try every combination, newest first, and keep the first that works.

```python
def vkey(v):
    m = re.fullmatch(r"(\d+(?:\.\d+)*)(?:(a|b|rc)(\d+))?", v)
    release = [int(x) for x in m.group(1).split(".")]
    release += [0] * (4 - len(release))
    pre = (PRE[m.group(2)], int(m.group(3))) if m.group(2) else (3, 0)
    return (tuple(release), pre)

def satisfies(v, spec):
    for part in [p.strip() for p in spec.split(",") if p.strip()]:
        op, target = re.fullmatch(r"(~=|==|!=|>=|<=|>|<)\s*(\S+)", part).groups()
        if target.endswith(".*"):
            prefix = target[:-2].split(".")
            same = [str(x) for x in vkey(v)[0][:len(prefix)]] == prefix
            if (op == "==") != same:
                return False
            continue
        a, b = vkey(v), vkey(target)
        if op == "~=":
            head = target.split(".")[:-1]
            if not (a >= b and [str(x) for x in a[0][:len(head)]] == head):
                return False
        elif not {"==": a == b, "!=": a != b, ">=": a >= b, "<=": a <= b, ">": a > b, "<": a < b}[op]:
            return False
    return True

def consistent(choice, index, root):
    for p, spec in root.items():
        if p in choice and not satisfies(choice[p], spec):
            return False
    for p, v in choice.items():
        for dep, spec in index[p][v].items():
            if dep in choice and not satisfies(choice[dep], spec):
                return False
    return True


def resolve(index, root):
    pkgs = sorted(index)
    options = [sorted(index[p], key=vkey, reverse=True) for p in pkgs]
    for combo in itertools.product(*options):
        choice = dict(zip(pkgs, combo))
        if consistent(choice, index, root):
            return choice
    return "conflict"
```

**Cost:** O(product of versions) time, O(p) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Backtracking: choose package by package, undo when a constraint breaks.

```python
def vkey(v):
    m = re.fullmatch(r"(\d+(?:\.\d+)*)(?:(a|b|rc)(\d+))?", v)
    release = [int(x) for x in m.group(1).split(".")]
    release += [0] * (4 - len(release))
    pre = (PRE[m.group(2)], int(m.group(3))) if m.group(2) else (3, 0)
    return (tuple(release), pre)

def satisfies(v, spec):
    for part in [p.strip() for p in spec.split(",") if p.strip()]:
        op, target = re.fullmatch(r"(~=|==|!=|>=|<=|>|<)\s*(\S+)", part).groups()
        if target.endswith(".*"):
            prefix = target[:-2].split(".")
            same = [str(x) for x in vkey(v)[0][:len(prefix)]] == prefix
            if (op == "==") != same:
                return False
            continue
        a, b = vkey(v), vkey(target)
        if op == "~=":
            head = target.split(".")[:-1]
            if not (a >= b and [str(x) for x in a[0][:len(head)]] == head):
                return False
        elif not {"==": a == b, "!=": a != b, ">=": a >= b, "<=": a <= b, ">": a > b, "<": a < b}[op]:
            return False
    return True

def consistent(choice, index, root):
    for p, spec in root.items():
        if p in choice and not satisfies(choice[p], spec):
            return False
    for p, v in choice.items():
        for dep, spec in index[p][v].items():
            if dep in choice and not satisfies(choice[dep], spec):
                return False
    return True


def resolve(index, root):
    pkgs = sorted(index)
    def go(i, choice):
        if i == len(pkgs):
            return dict(choice)
        for v in sorted(index[pkgs[i]], key=vkey, reverse=True):
            choice[pkgs[i]] = v
            if consistent(choice, index, root):
                found = go(i + 1, choice)
                if found:
                    return found
            del choice[pkgs[i]]
        return None
    return go(0, {}) or "conflict"
```

**Cost:** O(product) worst case, far less in practice time, O(p) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Try every combination, newest first, and keep the first that works. That costs O(product of versions) time and O(p) space."

"I can do better: Backtracking: choose package by package, undo when a constraint breaks. That's O(product) worst case, far less in practice time and O(p) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `index = {'a': {'2.0': {'b': '<2'}, '1.0': {}}, 'b': {'2.0': {}, '1.0': {}}}, root = {}`

1. go() returns {'a': '2.0', 'b': '1.0'}
2. go() returns {'a': '2.0', 'b': '1.0'}
3. go() returns {'a': '2.0', 'b': '1.0'}
4. resolve() returns {'a': '2.0', 'b': '1.0'}

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/engineering-toolbox/#/q/packaging/resolve-dependencies)

---
# Part 4: Interview questions, out loud

Cover the answer, **say yours out loud**, then open it and compare. Aim for two or three clear sentences, then an example.

<details>
<summary><b>1. Why use a virtual environment?</b></summary>

So each project has its own package versions and can't break another project or the system Python, and so the environment can be rebuilt from its lockfile.

</details>

<details>
<summary><b>2. requirements.txt vs pyproject.toml vs lockfile?</b></summary>

pyproject.toml declares what your project needs (ranges); a lockfile pins every package, including transitive ones, to exact versions; requirements.txt is the older format used for either.

</details>

<details>
<summary><b>3. What does ~=1.4.2 mean?</b></summary>

Compatible release: at least 1.4.2 but still 1.4.\* — patch updates only. `~=1.4` allows any 1.x from 1.4 upwards.

</details>

<details>
<summary><b>4. How do you handle a dependency conflict?</b></summary>

Read which packages constrain the shared dependency, upgrade or downgrade one of them so their ranges overlap, or isolate them in separate services or environments.

</details>

<details>
<summary><b>5. Where should secrets go?</b></summary>

Environment variables injected at runtime or a secret manager (AWS Secrets Manager, Vault). Never in code or Git; keep a .env.example with dummy values.

</details>

<details>
<summary><b>6. Should libraries pin exact versions?</b></summary>

No — pinning in a library causes conflicts for everyone using it. Libraries declare compatible ranges; applications pin via lockfiles.

</details>


---
# Check yourself before moving on

You're ready for the next notebook when you can tick these **without looking**:

- [ ] I can explain: Environments and dependencies
- [ ] I can explain: Versions and specifiers
- [ ] I can explain: How installers resolve and install
- [ ] I can explain: Keeping dependencies healthy and config clean

**Revision drill.** Come back in a few days, set a 25-minute timer per question, and solve these from a blank cell:

- [ ] [Q16](#q16) Configuration Precedence
- [ ] [Q17](#q17) Parse a .env File
- [ ] [Q18](#q18) Safe Automatic Upgrades
- [ ] [Q19](#q19) Is This Release Breaking?
- [ ] [Q20](#q20) Resolve Dependencies

Spaced repetition beats re-reading: redoing a question after you've partly forgotten it is what makes it stick.